# Regressão Logística — contrato, custo de ajuste e espaço de busca

Este notebook é a frente de **modelos interpretáveis** da comparação da Seção 4.4 (ART.7). Ele
começa onde o `modelagem.ipynb` da Seção 4.3 termina: consome o contrato de dados já pronto e não
repete nenhuma etapa de limpeza, integração ou pré-processamento.

Princípios adotados, os mesmos do notebook da Seção 4.3:

- consumir a matriz do contrato, sem reprocessar as fontes brutas nem reajustar transformador;
- importar de `src/` em vez de colar código na célula, para que notebook e `pytest` exercitem
  exatamente o mesmo módulo;
- nenhuma partição criada aqui. A divisão por data e por Cliente já veio congelada, e recriá-la
  neste notebook produziria uma segunda divisão que ninguém acordou.

**Seção 1, entregue pelo card #206.** Mede quanto custa um ajuste da Regressão Logística sobre o
contrato e se ela converge. Os dois números existem para dimensionar a grade de `GridSearchCV`
do card #207 antes de a busca começar, e não para avaliar qualidade: nenhuma métrica de
desempenho sai desta seção.

## 1. Custo de um ajuste e convergência (card #206)

### 1.1. Dependências

As versões de `scikit-learn` e `scipy` estão fixas em `requirements.txt` a partir do Sprint 04.
A fixação não é burocracia: o número de iterações até convergir, medido na Seção 1.4, depende da
implementação do solver, e um piso `>=` deixaria o Colab instalar outra minor e devolver um custo
diferente do que a grade do card #207 assume.

In [ ]:
!pip install -q -r requirements.txt

import sys
import time
from pathlib import Path

import pandas as pd
import scipy
import sklearn

# Inclui src/ no caminho de importação tanto no projeto local quanto no Colab.
# O break evita que duas cópias dos módulos fiquem no caminho ao mesmo tempo, o
# que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from fumaca_logistica import medir_ajuste_unico, menor_max_iter_que_converge
from matriz import preparar_matriz, resumo_da_matriz

print("scikit-learn", sklearn.__version__)
print("scipy", scipy.__version__)

As versões impressas têm que ser `scikit-learn 1.9.1` e `scipy 1.18.1`, as mesmas declaradas em
`requirements.txt`. Divergência aqui invalida a comparação de custo das seções seguintes: o tempo
medido passaria a descrever outra implementação do `lbfgs`, e a conta que dimensiona a grade do
card #207 deixaria de valer.

### 1.2. Carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a
célula falha com a instrução do que fazer, em vez de seguir com uma base vazia. A base não é
versionada, conforme o Termo de Abertura, que veda publicar dados do parceiro.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(
        "base_analitica.parquet não encontrada. Rode notebooks/pre-processamento.ipynb "
        f"ou ajuste o caminho. Procurado em: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas, {df.shape[1]} colunas")
print(f"Clientes distintos: {df['ID_GOLDENRECORD'].nunique():,}")

A base tem **484.915 linhas e 51 colunas**, com **407.139 valores distintos de
`ID_GOLDENRECORD`**. A distância entre os dois números é o motivo de a validação cruzada do card
#210 precisar agrupar por Cliente: há Cliente com mais de uma resposta, e a Hipótese 4 da Seção
4.2.4 mostrou que quem detratou uma vez volta a detratar com chance 4,61 vezes maior. Um Cliente
presente em dois folds faria o modelo reconhecer a pessoa em vez de aprender o fenômeno.

### 1.3. Matriz do contrato

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam acontecer
nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features do score
pós-viagem e ajustar o pré-processador **apenas no treino**.

Esta é a única fonte de `X`, `y`, `grupos` e do transformador em todo o notebook. As datas de
corte vêm do registro de decisão do #127, não são escolhidas aqui.

In [ ]:
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

inicio = time.perf_counter()
preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)
print(f"preparar_matriz: {time.perf_counter() - inicio:.1f}s")

matriz_treino = preparo["matrizes"]["treino"]
alvo_treino = preparo["y"]["treino"]
grupos_treino = preparo["grupos"]["treino"]

print(f"matriz de treino: {matriz_treino.shape[0]:,} linhas x {matriz_treino.shape[1]} colunas")
print(f"colunas na saída do pré-processador: {len(preparo['preprocessador'].get_feature_names_out())}")
print(f"Clientes no treino: {grupos_treino.nunique():,}")
print(f"prevalência de Detrator no treino: {alvo_treino.mean():.4f}")

resumo_da_matriz(preparo)

O contrato entrega uma matriz de treino com **341.962 linhas e 38 colunas**, cobrindo **307.510
Clientes**, e a prevalência de Detrator no treino é de **0,2043**. As 38 colunas da matriz são
exatamente as 38 que `get_feature_names_out` reporta, o que confirma que nada foi transformado
fora do contrato. O preparo inteiro custa cerca de **5,3 segundos**, desprezível diante do ajuste
medido a seguir.

A prevalência de 0,2043 é o número que o card #207 usa para justificar o eixo `class_weight` da
grade: com quatro não Detratores para cada Detrator, o modelo sem reponderação tende a acertar a
classe majoritária e a errar justamente quem a operação precisa encontrar.

### 1.4. Ajuste único: quanto custa e se converge

`medir_ajuste_unico` ajusta `LogisticRegression` **uma vez** sobre a matriz do contrato e devolve
tempo, iterações e convergência. Ele não monta `Pipeline`, que é o card #208, não reparticiona e
não calcula métrica: o número que interessa aqui é o relógio.

O aviso de convergência é capturado em vez de silenciado. Um ajuste que para no limite de
iterações entrega coeficiente provisório, e o card #212 lê exatamente esses coeficientes como
odds ratio.

In [ ]:
medicao_padrao = medir_ajuste_unico(matriz_treino, alvo_treino)

for chave in ("max_iter", "solver", "n_iter", "convergiu", "segundos"):
    print(f"{chave}: {medicao_padrao[chave]}")

if medicao_padrao["aviso"]:
    print("\naviso de convergência:")
    print(medicao_padrao["aviso"])

**O achado do card: a Regressão Logística não converge no `max_iter` padrão.** Com
`max_iter=100`, `solver='lbfgs'` e semente 42, o ajuste consome as 100 iterações, leva cerca de
**8,0 segundos** e emite `ConvergenceWarning` com `STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT`.

**A escala das features não é a causa, e isso importa.** O aviso do scikit-learn sugere escalonar
os dados, mas o contrato já faz isso: `matriz._montar_preprocessador` aplica `RobustScaler` nas
numéricas e codificação one-hot nas categóricas, então o `lbfgs` já recebe colunas em ordem de
grandeza comparável. Montar um escalonamento novo por fora do contrato para tentar acelerar a
convergência produziria uma segunda matriz e quebraria a comparação da Seção 4.4, que exige que
os quatro modelos consumam a mesma. O caminho correto é subir o `max_iter`, que é o que a Seção
1.5 mede.

A consequência para o card #207 é direta: `max_iter` não pode ficar no default na grade de busca,
senão toda combinação testada seria avaliada com coeficiente truncado.

### 1.5. Qual `max_iter` basta, e quanto ele custa

`menor_max_iter_que_converge` sobe o limite pela escala 100, 200, 400, 800, 1600 e para na
primeira tentativa que converge. Dobrar a cada passo mantém o custo total de procurar da ordem do
último ajuste, em vez da soma de uma varredura fina.

A trilha de tentativas é registrada de propósito: **só o tempo do ajuste que converge entra na
conta de custo da grade do card #207.** Os ajustes truncados param antes e custam menos, e usá-los
subestimaria a busca.

In [ ]:
busca_max_iter = menor_max_iter_que_converge(matriz_treino, alvo_treino)

for tentativa in busca_max_iter["tentativas"]:
    estado = "converge" if tentativa["convergiu"] else "trunca"
    print(f"max_iter={tentativa['max_iter']:>5} | n_iter={tentativa['n_iter']:>5} | "
          f"{tentativa['segundos']:>6.1f}s | {estado}")

escolhido = busca_max_iter["escolhido"]
print(f"\nescolhido: max_iter={escolhido['max_iter']}, "
      f"converge em {escolhido['n_iter']} iterações, {escolhido['segundos']:.1f}s")

A escala mostra onde está o limite:

| `max_iter` | iterações usadas | tempo | resultado |
|---:|---:|---:|---|
| 100 | 100 | 9,2s | trunca |
| 200 | 200 | 18,2s | trunca |
| 400 | 400 | 36,5s | trunca |
| 800 | **534** | **47,4s** | **converge** |

**A Regressão Logística converge em 534 iterações e custa cerca de 47 segundos por ajuste**, medido
nesta máquina (WSL2 local, CPU, scikit-learn 1.9.1 e scipy 1.18.1). No Colab o número muda com a
máquina sorteada, e por isso o que se transporta para o card #207 não é o segundo exato e sim a
ordem de grandeza: **um ajuste que converge custa dezenas de segundos, não centenas de
milissegundos.**

Até 400 iterações o tempo cresce proporcionalmente ao limite, porque o ajuste é truncado e gasta
todas as iterações disponíveis. Em 800 ele para antes, em 534, e é por isso que o tempo não dobra
de novo.

**O que este card entrega ao card #207**, que é quem declara a grade:

- `max_iter` precisa entrar fixado em pelo menos **800** na grade, e não como default;
- a conta de custo da busca é **47s por ajuste x número de folds x número de combinações**. Com os
  5 folds da validação por Cliente, cada combinação custa cerca de 4 minutos em série, o que
  limita a grade a poucas dezenas de combinações dentro de uma sessão do Colab;
- a prevalência de **0,2043** é o dado que sustenta o eixo `class_weight` da grade.

O mesmo número vai para a Luiza, que precisa dele para dimensionar a grade da Árvore de Decisão
sob o mesmo critério.

## 2. Espaço de busca da Regressão Logística (card #207)

Esta seção declara a grade que o card #210 vai executar no `GridSearchCV`, **antes** de qualquer
busca rodar. A ordem importa para o barema: grade declarada e justificada antes da execução é
evidência de método, enquanto grade ajustada depois de ver o resultado é tentativa e erro contada
como se fosse método.

A grade vive em `src/espaco_busca_logistica.py`, não numa célula daqui. Assim o `pytest` confere
que toda combinação declarada é válida, e o card #210 importa exatamente o mesmo objeto em vez de
reescrevê-lo.

### 2.1. O que a grade busca, e o que fica fixo

Três eixos variam. Dois parâmetros ficam fixos, e essa também é uma decisão do card:

| Parâmetro | Valores | Por que |
|---|---|---|
| `C` | 0.01, 0.1, 1.0, 10.0 | Escala logarítmica, de regularização forte a fraca. São 38 colunas para 341.962 linhas de treino, regime em que a verossimilhança domina a penalidade: quatro pontos separados por uma ordem de grandeza mostram se `C` tem efeito, sem gastar folds na terceira casa decimal |
| `l1_ratio` | 0.0, 1.0 | L2 pura contra L1 pura. A L1 zera coeficiente, e coeficiente zerado sai da leitura de odds ratio do card #212 |
| `class_weight` | `None`, `"balanced"` | A prevalência de Detrator no treino é 0,2043, quatro não Detratores para cada Detrator |
| `solver` | fixo: `lbfgs` na L2, `liblinear` na L1 | **Não é eixo de busca** |
| `max_iter` | fixo em 1600 | **Não é eixo de busca** |

**Por que `penalty` não aparece.** Ele está depreciado desde o scikit-learn 1.8 e é removido na
1.10. Quem expressa a penalidade agora é `l1_ratio`, com `0.0` para L2 e `1.0` para L1. Declarar a
grade com o parâmetro antigo faria a busca inteira emitir `FutureWarning` e quebraria na próxima
atualização do `requirements.txt`.

**Por que `solver` não é eixo de busca.** A verossimilhança da Regressão Logística com penalidade
L1 ou L2 é convexa: dois solvers que convergem chegam ao mesmo ótimo. O que muda entre eles é o
tempo, não o modelo. Buscar sobre o solver gastaria folds para escolher entre respostas iguais.
Ele fica fixo em cada subgrade, escolhido pela penalidade que aquela subgrade usa.

**Por que `max_iter` não é eixo de busca.** Buscar sobre o limite de iterações compararia quem
parou antes com quem terminou, e o card #212 lê esses coeficientes como odds ratio. O valor fixo
vem da medição da Seção 2.3.

**Por que a grade é uma lista de subgrades.** Nem toda combinação de `l1_ratio` com `solver`
existe: `lbfgs` só aceita L2 e levanta `ValueError` com L1. Um produto cartesiano único geraria
esse par inválido, e a busca morreria no meio depois de já ter gasto folds. A lista de subgrades é
como o `GridSearchCV` recebe espaços em que os eixos não são independentes.

In [ ]:
import pandas as pd

from espaco_busca_logistica import (
    GRADE_LOGISTICA,
    MAX_ITER,
    SEGUNDOS_POR_AJUSTE,
    combinacoes,
    custo_estimado,
    medir_ajuste,
)
from validacao import N_FOLDS

tabela_grade = pd.DataFrame(combinacoes())
print(f"{len(tabela_grade)} combinações, {N_FOLDS} folds, max_iter fixo em {MAX_ITER}")
tabela_grade.groupby(["solver", "l1_ratio"], dropna=False).size().rename("combinações")

São **16 combinações**, oito em cada subgrade: 4 valores de `C` x 2 de `class_weight`. O número de
folds vem de `validacao.N_FOLDS`, que é o `GroupKFold` de cinco folds por Cliente já usado na
Seção 4.3, e não um valor escolhido aqui.

### 2.2. `class_weight` e a prevalência de 0,2043

Com quatro não Detratores para cada Detrator, um modelo sem reponderação tende a acertar a classe
majoritária: ele erra pouco no total justamente porque quase nunca aponta Detrator, que é quem a
operação precisa encontrar antes do voo. `class_weight="balanced"` reponderar na proporção inversa
da frequência e empurra o modelo na direção oposta, achando mais Detratores ao custo de mais falso
positivo.

Os dois erros não custam o mesmo para a Azul, e por isso o eixo é medido em vez de decidido no
padrão da biblioteca: a escolha final sai da métrica oficial da comparação, no card #210, não
deste card.

### 2.3. Quanto custa a busca

A conta que dimensiona a grade é **combinações x folds x tempo de um ajuste**, e o tempo precisa
ser medido, não estimado de catálogo. O card #206 mediu um ajuste só, no ponto de partida da
biblioteca (`C=1.0`, sem reponderação): 47 s e 534 iterações. Esse número não cobre a grade, porque
o tempo muda com `C` e com `class_weight`.

A célula abaixo mede os **extremos** de cada subgrade, que é o que dá teto à conta.

In [ ]:
piores_casos = [
    {"C": C, "l1_ratio": l1, "class_weight": peso, "solver": solver, "max_iter": MAX_ITER}
    for solver, l1 in (("lbfgs", 0.0), ("liblinear", 1.0))
    for peso in (None, "balanced")
    for C in (0.01, 10.0)
]

medicoes = pd.DataFrame([
    {**{k: v for k, v in caso.items() if k != "max_iter"},
     **{k: v for k, v in medir_ajuste(matriz_treino, alvo_treino, caso).items()
        if k in ("segundos", "n_iter", "convergiu", "coeficientes_nulos")}}
    for caso in piores_casos
])
medicoes.sort_values("segundos", ascending=False)

O resultado desmonta duas suposições que pareciam seguras:

| solver | `C` | `class_weight` | tempo | iterações | converge |
|---|---:|---|---:|---:|---|
| `lbfgs` | 10.0 | `balanced` | 76,9 s | 838 | **não, com `max_iter=800`** |
| `lbfgs` | 0.01 | `None` | 72,2 s | 794 | sim |
| `lbfgs` | 10.0 | `None` | 42,1 s | 508 | sim |
| `lbfgs` | 0.01 | `balanced` | 36,9 s | 469 | sim |
| `liblinear` | 0.01 | `None` | 7,8 s | 20 | sim |
| `liblinear` | 0.01 | `balanced` | 5,0 s | 21 | sim |
| `liblinear` | 10.0 | `None` | 1,4 s | 15 | sim |
| `liblinear` | 10.0 | `balanced` | 1,1 s | 14 | sim |

**Primeira: o limite de 800 iterações herdado do card #206 não cobre a grade.** A combinação
`C=10.0` com `class_weight="balanced"` precisa de **838 iterações**, então com 800 ela truncaria e
entregaria coeficiente provisório para o card #212 ler como odds ratio. O `max_iter` da grade sobe
para **1600**, o degrau seguinte da mesma escala que o #206 usou. Subir o teto não encarece a
busca: `max_iter` é limite, não trabalho fixo, e o `lbfgs` para quando converge. As combinações que
já convergiam em 469 ou 508 iterações continuam custando o mesmo.

**Segunda: o tempo não cresce com `C`.** Sem reponderação o extremo caro é `C=0.01` (72,2 s contra
42,1 s), e com `balanced` a ordem se inverte (36,9 s contra 76,9 s). Escolher um extremo por
intuição erraria a conta por quase o dobro, e sempre é possível errar para menos, que é o lado que
promete uma busca que não cabe na sessão.

**Terceira, que não era suposição e sim descoberta útil:** o `liblinear` é de uma ordem de grandeza
mais barato. A metade L1 da grade custa menos do que um único ajuste da metade L2.

In [ ]:
custo = custo_estimado(SEGUNDOS_POR_AJUSTE, n_folds=N_FOLDS)

print(f"{custo['combinacoes']} combinações x {custo['n_folds']} folds + refit = {custo['ajustes']} ajustes")
for solver, segundos in custo["por_solver"].items():
    print(f"  {solver:>10}: {segundos / 60:5.1f} min")
print(f"  {'refit':>10}: {custo['refit'] / 60:5.1f} min")
print(f"\ntotal: {custo['segundos'] / 60:.1f} min em série, "
      f"{custo['segundos'] / 60 / 2:.1f} min com os 2 núcleos do Colab (n_jobs=-1)")

**A busca cabe na sessão do Colab.** São 81 ajustes: 51,3 min na metade `lbfgs`, 5,2 min na metade
`liblinear` e 1,3 min de `refit`, ou **57,7 min em série**. Com `n_jobs=-1` nos dois núcleos que o
Colab entrega, cerca de **29 min**, bem dentro do limite de inatividade de 90 min.

A conta é teto por dois motivos, os dois a favor: cada ajuste da busca roda sobre quatro quintos do
treino, porque o quinto restante é o fold de validação, e a grade tem combinações muito mais
baratas do que o pior caso usado na multiplicação.

**O que este card entrega ao card #210**, que é quem executa a busca:

- a grade em `src/espaco_busca_logistica.py`, importável, com `grade_com_prefixo()` para o nome de
  passo do `Pipeline` do card #208;
- `max_iter=1600` fixo, medido contra o pior caso e não herdado;
- a conta de custo acima, para a execução ser comparada com o previsto em vez de descoberta no
  meio.

O mesmo critério de dimensionamento vai para a Luiza, que declara a grade da Árvore de Decisão:
medir os extremos do espaço, não o ponto de partida da biblioteca.

## 3. Pipeline sobre o contrato (card #208)

As Seções 1 e 2 mediram o custo de um ajuste e declararam a grade. Falta o objeto que a busca do
card #210 vai receber: **um único `Pipeline`** que encadeia o pré-processador do contrato e a
`LogisticRegression`.

Enquanto os dois vivem soltos, quem ajusta decide onde ajustar, e a escolha mais cômoda é
justamente a que vaza. As Seções 1 e 2 mediram sobre `preparo["matrizes"]["treino"]`, que já saiu
do pré-processador ajustado no treino inteiro. Para *medir tempo* isso é correto e proposital: o
número que aqueles cards queriam era o relógio do ajuste, sem a codificação das categóricas no
meio. Para *buscar hiperparâmetro* deixa de ser: a mediana da imputação e os quartis do
`RobustScaler` passariam a carregar as linhas que cada fold usa como validação, e a métrica de
validação subiria sem que nada tivesse melhorado.

Dentro de um `Pipeline`, essa escolha deixa de existir. O `GridSearchCV` chama `fit` no objeto
inteiro a cada fold, e o `ColumnTransformer` é reajustado ali, sobre os quatro quintos daquele
fold. Não é uma disciplina que alguém precisa lembrar de seguir: é o que o card #209 trava por
teste.

### 3.1. O que o pipeline encadeia, e o que ele não decide

O pré-processador **não é remontado aqui**. `criar_pipeline` recebe o objeto que
`preparar_matriz` devolveu em `preparo["preprocessador"]` e faz `clone` dele: mesma especificação
(imputação, `RobustScaler`, `OneHotEncoder`, mesmas listas de colunas numéricas e categóricas),
sem o estado ajustado. Remontar o `ColumnTransformer` nesta célula criaria uma segunda definição
do contrato, e duas definições divergem sem ninguém perceber.

O `clone` também descarta de propósito o ajuste que `preparar_matriz` já fez no treino: o pipeline
precisa nascer virgem para que o primeiro `fit` seja o do fold.

Dois valores ficam fixos, e nenhum dos dois é escolha de hiperparâmetro deste card:

| Parâmetro | Valor | Por que aqui |
|---|---|---|
| `max_iter` | 1600 | Vem de `espaco_busca_logistica.MAX_ITER`, medido na Seção 2.3 contra o pior caso da grade. No padrão da biblioteca (100) o ajuste trunca, e coeficiente truncado é o que o card #212 leria como odds ratio |
| `random_state` | 42 | Mesma semente de `congelamento.SEMENTE`. O `liblinear` embaralha internamente: sem ela, duas execuções da mesma combinação dariam coeficientes diferentes |

`C`, `l1_ratio` e `class_weight` ficam no padrão da biblioteca. Quem escolhe os valores é a busca
do card #210 sobre a grade do card #207, não esta seção.

In [ ]:
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import ParameterGrid
from sklearn.utils.validation import check_is_fitted

from espaco_busca_logistica import grade_com_prefixo
from pipeline_logistica import (PASSO_MODELO, PASSO_PREPARO, ajustar_no_treino,
                                criar_pipeline)

pipeline = criar_pipeline(preparo["preprocessador"])
estimador = pipeline.named_steps[PASSO_MODELO]

print("passos:", [nome for nome, _ in pipeline.steps])
print(f"max_iter={estimador.max_iter}, random_state={estimador.random_state}, "
      f"C={estimador.C}, l1_ratio={estimador.l1_ratio}, class_weight={estimador.class_weight}")

# O pré-processador do pipeline tem que nascer NÃO ajustado, senão o primeiro
# `fit` de cada fold não seria o que define a mediana e a escala daquele fold.
try:
    check_is_fitted(pipeline.named_steps[PASSO_PREPARO])
    print("\nATENÇÃO: o pré-processador do pipeline já nasceu ajustado")
except NotFittedError:
    print("\npré-processador do pipeline: não ajustado, como esperado")

# A grade do card #207 endereça o passo do pipeline sem ser reescrita à mão.
combinacoes_prefixadas = list(ParameterGrid(grade_com_prefixo(PASSO_MODELO)))
print(f"\n{len(combinacoes_prefixadas)} combinações prefixadas, prontas para o card #210")
print("exemplo:", combinacoes_prefixadas[0])

O pipeline tem os dois passos, `preparo` e `modelo`, e o pré-processador nasce não ajustado. Os
hiperparâmetros saem no padrão da biblioteca (`C=1.0`, `l1_ratio=0.0`, `class_weight=None`),
como este card pede.

A última verificação é a que amarra este card ao #210: `grade_com_prefixo(PASSO_MODELO)` devolve
as mesmas **16 combinações** da Seção 2.1, agora com as chaves no formato `modelo__parâmetro`, que
é o único que o `GridSearchCV` reconhece quando o estimador é um `Pipeline`. O nome do passo vem
da constante `PASSO_MODELO`, não de uma string repetida nos dois arquivos: trocar o nome em um
lugar só faria a busca varrer um parâmetro inexistente, e o erro só apareceria no primeiro ajuste,
depois de a busca já ter começado.

### 3.2. Ajuste único sobre a partição de treino

Agora o pipeline inteiro é ajustado uma vez sobre `preparo["x"]["treino"]`, a matriz **crua**,
com as colunas da allowlist ainda por transformar, e não `preparo["matrizes"]["treino"]`, que já
passou pelo pré-processador nas Seções 1 e 2. É essa diferença que faz o `ColumnTransformer` ser
ajustado de dentro do pipeline.

O ajuste aqui não é a busca: é o ponto de partida contra o qual o card #210 vai comparar o ganho
do melhor conjunto de hiperparâmetros.

In [ ]:
relato_ajuste = ajustar_no_treino(pipeline, preparo["x"]["treino"], preparo["y"]["treino"])

for chave in ("n_linhas", "colunas_de_entrada", "colunas_da_matriz",
              "n_iter", "max_iter", "convergiu", "segundos", "coeficientes_nulos"):
    print(f"{chave}: {relato_ajuste[chave]}")

if relato_ajuste["aviso"]:
    print("\naviso de convergência:")
    print(relato_ajuste["aviso"])

O ajuste de dentro do pipeline **converge**, em **534 iterações** e algo em torno de **45 s**
(WSL2 local, CPU, `scikit-learn` 1.9.1; duas execuções deram 41,6 s e 50,4 s). São as mesmas 534
iterações que a Seção 1.5 mediu sobre a matriz já transformada, e a coincidência é o resultado que
se queria: encadear o pré-processador não muda o problema de otimização que o `lbfgs` resolve, só
muda **quando** a transformação acontece.

As **14 colunas de entrada** viram **38 colunas de matriz**: a expansão do `OneHotEncoder` sobre
as categóricas mais os indicadores de ausência do `SimpleImputer`, exatamente a largura que a
Seção 1.3 já reportava. Nenhum coeficiente sai zerado, o que era esperado com a penalidade L2 do
padrão da biblioteca: quem zera coeficiente é a L1, e ela entra na grade do card #210.

### 3.3. Métrica de partida

O card pede que a métrica de partida saia da função `avaliar()` do card 05A.1 (#241), e **não** de
uma métrica escrita à parte. Esse é o ponto do card 05: as quatro duplas comparam os candidatos na
tabela do card 18A.1, e isso só funciona se todas lerem F2, Sensibilidade, Precisão Média e ROC-AUC
da mesma implementação.

Quando a Seção 3 foi escrita, o #241 ainda não estava em `develop` e esta seção ficou registrada
como pendente. Ele entrou depois, e a Seção 9 calcula a métrica de partida com ele, junto com a do
modelo otimizado, para que as duas apareçam lado a lado.

O consumo continua sendo o de `pipeline_logistica.metrica_de_partida`, que recebe a função como
argumento e repassa os vetores sem calcular nem renomear nada. A probabilidade que entra é a da
classe positiva, a coluna 1 de `predict_proba`: Precisão Média e ROC-AUC são calculadas sobre o
score contínuo, e passar o rótulo devolveria um número válido e errado.

## 4. Busca em grade da Regressão Logística (card #210)

A Seção 3 entregou o pipeline não ajustado. Esta seção passa esse objeto ao `GridSearchCV` e deixa
a busca escolher os hiperparâmetros que a Seção 2 declarou, sobre a validação cruzada por Cliente
do contrato.

Três travas valem para tudo o que segue, e nenhuma delas é escolha desta seção:

- **A validação é agrupada por `ID_GOLDENRECORD`**, e vem de `validacao.criar_folds`, não de um
  split montado aqui. O mesmo Cliente aparece em várias respostas, e um `StratifiedKFold` ingênuo
  colocaria o mesmo Cliente nos dois lados do fold: o modelo reconheceria o Cliente em vez de
  aprender o que leva à detração, e a métrica de validação subiria por vazamento.
- **A partição de teste não é tocada.** A busca recebe apenas `preparo["x"]["treino"]`. O teste
  fica para o card 13A.1 (#212), depois de os hiperparâmetros estarem escolhidos e congelados.
- **O pré-processamento acontece dentro de cada fold**, porque o que entra no `GridSearchCV` é o
  `Pipeline` da Seção 3, e não a matriz já transformada das Seções 1 e 2.

### 4.1. A grade reduzida, e por que ela foi reduzida

A Seção 2.3 estimou a grade inteira do card #207 em **58 minutos** de ajuste: 16
combinações vezes 5 folds, mais o refit, com o `lbfgs` respondendo por 51 desses minutos sozinho.
Não cabe na sessão de trabalho, e o card prevê o caso: reduzir a grade e registrar a redução, em
vez de interromper a execução no meio e reportar resultado parcial como final.

O corte é **em um eixo só**, o `C`, que vai de quatro valores para dois:

| Eixo | Card #207 | Aqui | Por quê |
|---|---|---|---|
| Penalidade | L2 (`lbfgs`) e L1 (`liblinear`) | as duas, intactas | É o eixo que o card #212 lê: a L1 zera coeficiente e a L2 não, e isso muda quais odds ratio existem para interpretar |
| `class_weight` | `None` e `balanced` | os dois, intactos | A prevalência de Detrator é 0,2043, e reponderar a classe é a decisão que mais mexe na Sensibilidade, que é o que o F2 privilegia |
| `C` | `[0.01, 0.1, 1.0, 10.0]` | `[0.1, 1.0]` | Restam os dois valores centrais. Os extremos varrem regularização muito forte e quase nenhuma, que são os cantos menos informativos para a leitura |

As 16 combinações viram **8**, e o custo estimado cai de 58 para cerca de 30 minutos de ajuste em série. A
grade é recortada a partir de `GRADE_LOGISTICA`, e não reescrita: reescrever criaria uma segunda
definição da grade, livre para divergir da do card #207 no dia em que ela mudar.

In [ ]:
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.model_selection import GridSearchCV

from espaco_busca_logistica import GRADE_LOGISTICA, combinacoes
from validacao import N_FOLDS, criar_folds

# Recorte de um eixo só, a partir da grade do card #207.
C_REDUZIDO = [0.1, 1.0]
GRADE_REDUZIDA = [{**bloco, "C": C_REDUZIDO} for bloco in GRADE_LOGISTICA]

# Os folds vêm do contrato, agrupados por Cliente. Não se monta split aqui.
folds = criar_folds(preparo["x"]["treino"], preparo["grupos"]["treino"])

busca = GridSearchCV(
    criar_pipeline(preparo["preprocessador"]),
    param_grid=grade_com_prefixo(PASSO_MODELO, GRADE_REDUZIDA),
    scoring=make_scorer(fbeta_score, beta=2),  # provisório, ver 4.4
    cv=folds,
    refit=True,
    n_jobs=5,  # um processo por fold; não altera o resultado, só o relógio
)

n_comb = len(combinacoes(GRADE_REDUZIDA))
print(f"{n_comb} combinações x {N_FOLDS} folds = {n_comb * N_FOLDS} ajustes, mais o refit")

### 4.2. Execução da busca

O `fit` abaixo recebe a matriz **crua** do treino, e não a transformada: é o pipeline que
transforma, dentro de cada fold. O aviso de convergência é capturado em vez de silenciado, pelo
mesmo motivo da Seção 3.2: uma combinação que trunca em `max_iter` entrega coeficiente provisório,
e isso precisa aparecer.

In [ ]:
import time
import warnings

with warnings.catch_warnings(record=True) as capturados:
    warnings.simplefilter("always")
    inicio = time.perf_counter()
    busca.fit(preparo["x"]["treino"], preparo["y"]["treino"])
    segundos_busca = time.perf_counter() - inicio

print("melhores hiperparâmetros:")
for chave, valor in sorted(busca.best_params_.items()):
    print(f"  {chave.replace(PASSO_MODELO + '__', '')}: {valor}")

print(f"\nmelhor F2 médio nos folds: {busca.best_score_:.4f}")
print(f"tempo total da busca: {segundos_busca:.1f}s ({segundos_busca / 60:.1f} min)")
print(f"ajustes feitos: {len(busca.cv_results_['params']) * N_FOLDS} + refit")
print(f"avisos de convergência: "
      f"{sum('onvergence' in a.category.__name__ for a in capturados)}")

A busca varreu as **8 combinações** em **40 ajustes mais o refit**, sobre
341.962 linhas e 307.510 Clientes do treino, e levou
**579 s (9,6 min)** de relógio com `n_jobs=5`. 40 ajustes
é exatamente 8 combinações vezes 5 folds, que é a conferência que o card pede: nenhum fold foi
pulado e nenhuma combinação ficou de fora.

Os hiperparâmetros vencedores:

| Parâmetro | Valor |
|---|---|
| `C` | 1.0 |
| `class_weight` | 'balanced' |
| `solver` | `liblinear` |
| penalidade | L1 (`l1_ratio=1.0`) |
| `max_iter` | 1600 |

**Melhor F2 médio na validação cruzada por Cliente: 0,5189.**

Avisos de convergência: nenhum, todas as combinações convergiram dentro do max_iter de 1600.

Uma ressalva que já cabe aqui, porque qualifica o número que está sendo registrado: a diferença de
F2 entre as quatro melhores combinações é de 0,0003,
enquanto o desvio entre os cinco folds da vencedora é de 0,0035, cerca de dez
vezes maior. O vencedor é o vencedor, mas a margem está dentro do ruído: trocar de vencedor entre
essas quatro não mudaria o desempenho de forma perceptível.

A tabela completa das combinações, com o desvio entre folds e a leitura de qual eixo moveu a
métrica, é o card 07A.4 (#211). Aqui ficam registrados apenas os três números que este card pede:
os parâmetros vencedores, a métrica do vencedor e o tempo total.

### 4.3. O `scoring` desta busca, e por que ela não precisou ser refeita

O card pede que o `scoring` da busca seja o do card 05A.1 (#241). Quando esta busca rodou, o #241
ainda não estava em `develop`, então ela usou `make_scorer(fbeta_score, beta=2)` montado aqui, e a
seção ficou registrada como pendente, com a ressalva de que a busca seria refeita se o `avaliar()`
oficial divergisse.

**Ele entrou, e não diverge.** A Seção 9 confere os dois na mesma partição: o `make_scorer` usado
na busca e o `avaliar()["F2"]` do #241 devolvem o mesmo valor até a décima casa, porque os dois
chamam `fbeta_score` com `beta=2` sobre o rótulo no limiar padrão. O ranking das oito combinações
é o mesmo com qualquer um dos dois, e **a busca não precisou ser reexecutada**.

O F2 continua sendo critério de busca e não métrica de comparação, como o protocolo do card #238
estabelece: a tabela do card 18A.1 é lida sobre Sensibilidade, Precisão Média e ROC-AUC.

### 4.4. Dois níveis de `TIER_VIAGEM` que o treino nunca vê

Levantado ao revisar a MR do card #208, e vale registrar aqui porque é a busca que herda o efeito.

`AZUL ONE` e `DIAMANTE UNIQUE` só aparecem na base a partir de **2025-10-24**. O corte de
validação é 2025-07-01, então as **1.353 linhas** desses dois tiers caem inteiras na partição de
**teste**: nem treino nem validação têm uma única linha delas.

O `OneHotEncoder` do contrato é `handle_unknown="ignore"`, então:

- o pré-processador ajustado no treino aprende **5 níveis** de `TIER_VIAGEM`, não 7, e a matriz
  sai com 38 colunas;
- **nenhum fold desta busca vê esses dois níveis**, e portanto nenhum hiperparâmetro foi escolhido
  levando-os em conta;
- no teste, as 1.353 linhas atravessam com o bloco de `TIER_VIAGEM` inteiro zerado. Não levanta
  erro e não emite aviso, então isso não apareceria sozinho em lugar nenhum.

Não é defeito do contrato: é o comportamento correto para categoria não vista, e a alternativa,
`handle_unknown="error"`, quebraria a pontuação do teste. O que não pode acontecer é o card 13A.1
(#212) ler o coeficiente de `TIER_VIAGEM` como se ele cobrisse os sete níveis, e o card 07B.1
(#213) interpretar isso em termos da operação. São justamente os Clientes de topo.

## 5. Hiperparâmetros vencedores e leitura da busca (card #211)

A Seção 4 deixou os vencedores impressos numa célula. Isso basta para ler o resultado hoje, e não
basta para o card 13A.1 (#212) reconstruir o modelo: quem abrir o notebook amanhã teria de
reexecutar a busca inteira só para recuperar quatro números.

Esta seção versiona a tabela da busca em `documents/extras/resultados/hiperparametros_logistica.json` e escreve a
leitura: qual eixo da grade moveu a métrica, qual ficou indiferente, e o quanto a escolha do
vencedor se sustenta.

O formato é o mesmo de `documents/extras/resultados/hiperparametros_candidato.json`, escrito no card #104 para a busca
da Seção 4.3: array JSON com uma entrada por combinação, ordenado pela métrica, com os
hiperparâmetros e o par média/desvio. É JSON e não CSV porque o projeto bloqueia arquivos CSV.

### 5.1. A tabela da busca, versionada

O tempo de ajuste fica **de fora** do arquivo de propósito. Ele muda entre duas execuções
idênticas, e um artefato versionado que produz diff sem o resultado ter mudado deixa de servir
como prova de que o resultado é o mesmo.

In [ ]:
from pathlib import Path

PASTA_RESULTADOS = next(p for p in (Path("documents/extras/resultados"), Path("../documents/extras/resultados")) if p.exists())
DESTINO_BUSCA = PASTA_RESULTADOS / "hiperparametros_logistica.json"

tabela_busca = (
    pd.DataFrame(busca.cv_results_)
    .assign(
        C=lambda d: d[f"param_{PASSO_MODELO}__C"],
        class_weight=lambda d: d[f"param_{PASSO_MODELO}__class_weight"],
        solver=lambda d: d[f"param_{PASSO_MODELO}__solver"],
        l1_ratio=lambda d: d[f"param_{PASSO_MODELO}__l1_ratio"],
        max_iter=lambda d: d[f"param_{PASSO_MODELO}__max_iter"],
        f2=lambda d: d["mean_test_score"],
        f2_desvio=lambda d: d["std_test_score"],
    )
    .sort_values("f2", ascending=False)
    .loc[:, ["C", "class_weight", "solver", "l1_ratio", "max_iter", "f2", "f2_desvio"]]
    .reset_index(drop=True)
)

tabela_busca.to_json(DESTINO_BUSCA, orient="records", indent=2, double_precision=10)
print(f"gravado em {DESTINO_BUSCA}")
tabela_busca

| # | penalidade | `C` | `class_weight` | F2 médio | desvio entre folds |
|---|---|---|---|---|---|
| 1 | L1 (`liblinear`) | 1,0 | `balanced` | **0,5189** | 0,0035 |
| 2 | L2 (`lbfgs`) | 0,1 | `balanced` | 0,5188 | 0,0033 |
| 3 | L1 (`liblinear`) | 0,1 | `balanced` | 0,5187 | 0,0034 |
| 4 | L2 (`lbfgs`) | 1,0 | `balanced` | 0,5186 | 0,0033 |
| 5 | L1 (`liblinear`) | 1,0 | `None` | 0,2753 | 0,0030 |
| 6 | L2 (`lbfgs`) | 1,0 | `None` | 0,2749 | 0,0033 |
| 7 | L1 (`liblinear`) | 0,1 | `None` | 0,2746 | 0,0031 |
| 8 | L2 (`lbfgs`) | 0,1 | `None` | 0,2739 | 0,0039 |

### 5.2. Um eixo decide, os outros dois não

**`class_weight` é o único eixo que move a métrica.** As quatro combinações com `balanced` ficam
em 0,5186 a 0,5189; as quatro com `None`, em 0,2739 a 0,2753. A diferença é de **0,244 em F2**,
quase o dobro do valor que as piores alcançam. E é o resultado esperado, não uma surpresa: a
prevalência de Detrator no treino é 0,2043, o F2 pesa Sensibilidade quatro vezes mais que
Precisão, e um modelo que não repondera a classe minoritária protege a classe errada.

**`C` e a penalidade são indiferentes.** Dentro do bloco `balanced`, o intervalo inteiro das
quatro combinações é de **0,0003**. Trocar a regularização de 0,1 para 1,0, ou L1 por L2, não
muda o desempenho de forma mensurável nesta base.

**A escolha do vencedor não é estável, e isso precisa acompanhar o número.** O desvio entre os
cinco folds da combinação vencedora é **0,0035**, cerca de dez vezes a distância que separa a
primeira da quarta colocada. Dito de outro jeito: a variação de F2 de um fold para o outro, dentro
da própria vencedora, é maior do que a vantagem dela sobre as concorrentes. O `C=1.0` com L1
venceu, mas qualquer uma das quatro primeiras entregaria o mesmo desempenho, e uma reexecução com
outra semente de particionamento poderia reordená-las.

**A L1 não zerou nenhum coeficiente.** Ajustando a vencedora sobre o treino, os 38 coeficientes
saem todos diferentes de zero. Em `scikit-learn` 1.9.1 o `l1_ratio=1.0` é L1 de verdade (o
`penalty` é que está depreciado), então a penalidade foi aplicada; ela é que não tem força para
zerar nada com `C=1.0` sobre 38 colunas que carregam sinal. Vale registrar porque o card 13A.1
(#212) lê os coeficientes como odds ratio: não há seleção de variável embutida aqui, todas as 38
entram na leitura.

### 5.3. Reconstruir sem repetir a busca

É isto que o card 13A.1 (#212) faz: lê o arquivo, monta o pipeline com os vencedores e ajusta uma
vez. Dez minutos de busca viram um `json.load`.

In [ ]:
import json

vencedores = json.loads(DESTINO_BUSCA.read_text())[0]
hiperparametros = {chave: valor for chave, valor in vencedores.items()
                   if chave not in ("f2", "f2_desvio", "max_iter")}

pipeline_final = criar_pipeline(preparo["preprocessador"],
                                max_iter=int(vencedores["max_iter"]),
                                **hiperparametros)
print(pipeline_final.named_steps[PASSO_MODELO])

## 6. Odds ratio por feature original (card #212)

A Regressão Logística entra na comparação como o modelo **interpretável**, e é esta seção que
cobra essa promessa: cada coeficiente vira um odds ratio e volta para a variável que a operação
conhece, não para o nome que o `ColumnTransformer` inventou.

O modelo é o reconstruído a partir de `documents/extras/resultados/hiperparametros_logistica.json`, com os vencedores
da busca do card #210, e não a configuração padrão da biblioteca.

**Uma ressalva que vale para a seção inteira:** odds ratio aqui descreve **associação que o modelo
usa para ordenar**, não causa. Dizer que um atraso "aumenta em 18% a chance de detração" é a
leitura errada; o certo é que, entre respostas parecidas nas demais variáveis, as que tiveram uma
hora a mais de atraso aparecem com chance 18% maior de serem detratoras **na ordenação que este
modelo produz**. O modelo não foi construído para responder o que acontece se a Azul mudar o
atraso, e nada aqui sustenta essa leitura.

### 6.1. Três transformações separam o coeficiente da variável

Entre a coluna do contrato e o número que o `LogisticRegression` produz existem três passos, e
ignorar qualquer um devolve um odds ratio válido e errado:

| Transformação | O que ela faz | O que muda na leitura |
|---|---|---|
| `OneHotEncoder` | quebra cada categórica em uma coluna por nível | 6 variáveis viram 23 colunas. O nome de saída é `categoricas__CANAL_COMPRA_OTHER`, e quem lê precisa da variável e do nível separados |
| `SimpleImputer(add_indicator=True)` | acrescenta um indicador por coluna com ausência | 7 colunas que **não medem a variável**, medem o fato de ela não ter sido medida |
| `RobustScaler` | divide pelo IQR do treino | `exp(coef)` passa a ser por IQR, não por unidade natural |

O terceiro tem uma armadilha própria nesta base. O `scale_` sai **1,0 em seis das oito
numéricas**, e por dois motivos diferentes:

- `ATRASO_CHEGADA`, `HIST_RESPOSTAS_ANTERIORES`, `HIST_DETRATOU_ANTES` e
  `HIST_TAXA_DETRACAO_ANTERIOR` têm IQR **genuinamente zero**: o miolo da distribuição é todo
  zero, porque a maioria dos voos chega no horário e a maioria das respostas não tem histórico;
- `ANTECEDENCIA_CANCELAMENTO` tem IQR 41 nos valores observados, mas só existe em voo cancelado.
  A imputação da mediana em 90% das linhas **achata o IQR a zero**, e o `scikit-learn` cai no
  fallback de 1,0.

Por isso a coluna se chama `escala_do_scaler` e não `iqr_do_treino`: ela nem sempre é o IQR, e o
nome errado esconderia justamente as colunas em que o escalonamento não aconteceu.

**Ordenar por `exp(coef)` é o mesmo erro numa terceira forma.** Aquela coluna mistura efeito por
IQR, por minuto, por dia e por nível, e comparar números em unidades diferentes não é ranking de
efeito. `ANTECEDENCIA_CANCELAMENTO` é medida **em dias** e vai de 0 a 400: 0,980 por dia parece
desprezível, e vale **0,288** no intervalo em que a variável de fato varia. Por isso a tabela
traz `odds_ratio_comparavel`, o efeito de percorrer o intervalo interdecil do treino (p10 a p90),
e é por ele que a ordenação acontece. O interdecil, e não o IQR, porque em quatro das oito
numéricas o IQR é zero e reduziria o efeito delas a 1,0 por construção.

A tabela traz também `n_observado`, o número de linhas do treino que sustentam cada coluna. Sem
ele não dá para saber se um odds ratio vem da base inteira ou de meio por cento dela.

In [ ]:
from odds_ratio import efeito_acumulado, odds_ratio

# `pipeline_final` vem da Seção 5.3, montado a partir do arquivo de hiperparâmetros.
# Aqui ele só é ajustado: remontar criaria uma segunda construção do mesmo objeto.
pipeline_final.fit(preparo["x"]["treino"], preparo["y"]["treino"])

tabela_odds = odds_ratio(pipeline_final, preparo["x"]["treino"])
print(f"{len(tabela_odds)} colunas da matriz, "
      f"{tabela_odds['feature_original'].nunique()} features originais")
tabela_odds

As 38 colunas, ordenadas pelo **efeito comparável**: o que acontece ao percorrer o intervalo p10 a
p90 da variável no treino, nas numéricas, e o próprio odds ratio nas colunas que já são 0/1. `n` é
quantas linhas do treino sustentam a coluna, de 341.962.

| # | Feature | Percurso | OR comparável | OR bruto | n |
|---|---|---|---|---|---|
| 1 | `HIST_TAXA_DETRACAO_ANTERIOR` | p10 a p90 = 1 | **4.245** | 4.245 | 34.452 |
| 2 | `ANTECEDENCIA_CANCELAMENTO` | p10 a p90 = 63 | **0.288** | 0.980 | 34.395 |
| 3 | ausência de `TEMPO_VOO` | 0 para 1 | **3.115** | 3.115 | 34.454 |
| 4 | ausência de `ANTECEDENCIA_CANCELAMENTO` | 0 para 1 | **0.384** | 0.384 | 307.567 |
| 5 | `CANAL_COMPRA` = OTHER | 0 para 1 | **2.363** | 2.363 | 2.118 |
| 6 | `ESTATISTICA_ATRASOSAIDA` | p10 a p90 = 35 | **1.986** | 1.193 | 307.567 |
| 7 | `TIER_VIAGEM` = SEM CADASTRO | 0 para 1 | **0.604** | 0.604 | 45.684 |
| 8 | `TIER_VIAGEM` = AZUL FIDELIDADE | 0 para 1 | **0.712** | 0.712 | 181.371 |
| 9 | ausência de `HIST_DETRATOU_ANTES` | 0 para 1 | **1.326** | 1.326 | 307.510 |
| 10 | `TIER_VIAGEM` = DIAMANTE | 0 para 1 | **1.326** | 1.326 | 43.707 |
| 11 | `SEGMENTO` = AZUL VIAGENS | 0 para 1 | **0.761** | 0.761 | 6.725 |
| 12 | `TIPO_ENTRETENIMENTO` = NAO_INFORMADO | 0 para 1 | **1.296** | 1.296 | 109.698 |
| 13 | `CANAL_COMPRA` = AEROPORTO | 0 para 1 | **1.295** | 1.295 | 1.185 |
| 14 | `HIST_DETRATOU_ANTES` | p10 a p90 = 1 | **1.289** | 1.289 | 34.452 |
| 15 | `TIPO_ENTRETENIMENTO` = AO VIVO | 0 para 1 | **0.789** | 0.789 | 111.545 |
| 16 | `TEMPO_VOO` | p10 a p90 = 365 | **1.193** | 1.070 | 307.508 |
| 17 | `VOO_TIPO` = CONEXÃO | 0 para 1 | **0.844** | 0.844 | 109.698 |
| 18 | `TIPO_ENTRETENIMENTO` = GRAVADO | 0 para 1 | **0.864** | 0.864 | 55.952 |
| 19 | `CANAL_COMPRA` = MOBILE | 0 para 1 | **0.870** | 0.870 | 83.561 |
| 20 | `CANAL_COMPRA` = WEB | 0 para 1 | **0.877** | 0.877 | 105.013 |
| 21 | `VOO_TIPO` = ESCALA | 0 para 1 | **1.133** | 1.133 | 1.010 |
| 22 | `SEGMENTO` = DEMAIS CLIENTES | 0 para 1 | **0.887** | 0.887 | 306.087 |
| 23 | ausência de `ESTATISTICA_ATRASOSAIDA` | 0 para 1 | **0.902** | 0.902 | 34.395 |
| 24 | ausência de `ATRASO_CHEGADA` | 0 para 1 | **0.907** | 0.907 | 34.395 |
| 25 | `SEGMENTO` = CORPORATIVO | 0 para 1 | **1.101** | 1.101 | 29.150 |
| 26 | `CANCELAMENTO_VOO` = True | 0 para 1 | **0.918** | 0.918 | 34.395 |
| 27 | `ATRASO_CHEGADA` | p10 a p90 = 29 | **1.085** | 1.003 | 307.567 |
| 28 | `HIST_RESPOSTAS_ANTERIORES` | p10 a p90 = 1 | **0.922** | 0.922 | 341.962 |
| 29 | `N_TRECHOS` | p10 a p90 = 1 | **0.950** | 0.950 | 307.567 |
| 30 | `TIPO_ENTRETENIMENTO` = NÃO TEM ENTRETENIMENTO | 0 para 1 | **1.051** | 1.051 | 64.767 |
| 31 | `CANAL_COMPRA` = AGENCY | 0 para 1 | **0.953** | 0.953 | 142.924 |
| 32 | `CANCELAMENTO_VOO` = False | 0 para 1 | **0.956** | 0.956 | 307.567 |
| 33 | `TIER_VIAGEM` = SAFIRA | 0 para 1 | **1.040** | 1.040 | 30.964 |
| 34 | `CANAL_COMPRA` = CALLCENTER | 0 para 1 | **1.037** | 1.037 | 7.161 |
| 35 | `TIER_VIAGEM` = TOPAZIO | 0 para 1 | **0.974** | 0.974 | 40.236 |
| 36 | ausência de `HIST_TAXA_DETRACAO_ANTERIOR` | 0 para 1 | **1.013** | 1.013 | 307.510 |
| 37 | `VOO_TIPO` = DIRETO | 0 para 1 | **0.991** | 0.991 | 231.254 |
| 38 | ausência de `N_TRECHOS` | 0 para 1 | **0.998** | 0.998 | 34.395 |

A diferença entre as duas colunas de OR é a razão de existir desta seção. `ANTECEDENCIA_CANCELAMENTO`
tem OR bruto 0,980, que parece nada, e OR comparável **0,288**, o segundo maior efeito da tabela:
ela é medida em dias e percorre 63 deles entre o p10 e o p90.

Nenhuma feature derivada da própria resposta de NPS aparece na lista. As três `HIST_*` são de
respostas **anteriores** do mesmo Cliente, construídas com anterioridade estrita em
`features.adicionar_historico`.

### 6.2. Comparando níveis contra uma referência declarada

O `OneHotEncoder` do contrato é `drop=None`, então todos os níveis entram e nenhum é o nível-base
da regressão. Isso não impede declarar uma referência na leitura: basta subtrair os coeficientes,
já que `exp(β_nível − β_referência)` é o odds ratio de um nível contra o outro.

A referência aqui é sempre **o nível mais frequente no treino**, que é a escolha que deixa a
comparação mais estável, porque o lado de baixo da razão é o que tem mais linhas.

In [ ]:
import numpy as np


def contra_referencia(tabela, feature):
    """Odds ratio de cada nível contra o nível mais frequente da mesma variável."""
    bloco = tabela[(tabela["feature_original"] == feature)
                   & (tabela["tipo"] == "categorica")].copy()
    referencia = bloco.loc[bloco["n_observado"].idxmax()]
    bloco["referencia"] = referencia["nivel"]
    bloco["or_vs_referencia"] = np.exp(bloco["coeficiente"] - referencia["coeficiente"])
    return (bloco.loc[bloco["nivel"] != referencia["nivel"],
                      ["feature_original", "nivel", "referencia", "or_vs_referencia", "n_observado"]]
            .sort_values("or_vs_referencia", ascending=False))


categoricas = tabela_odds.loc[tabela_odds["tipo"] == "categorica", "feature_original"].unique()
odds_por_referencia = pd.concat([contra_referencia(tabela_odds, f) for f in categoricas],
                                ignore_index=True)
odds_por_referencia

| Variável | Referência (n) | Demais níveis |
|---|---|---|
| `CANAL_COMPRA` | AGENCY (142.924) | OTHER **2.48** · AEROPORTO **1.36** · CALLCENTER **1.09** · WEB **0.92** · MOBILE **0.91** |
| `TIER_VIAGEM` | AZUL FIDELIDADE (181.371) | DIAMANTE **1.86** · SAFIRA **1.46** · TOPAZIO **1.37** · SEM CADASTRO **0.85** |
| `SEGMENTO` | DEMAIS CLIENTES (306.087) | CORPORATIVO **1.24** · AZUL VIAGENS **0.86** |
| `TIPO_ENTRETENIMENTO` | AO VIVO (111.545) | NAO_INFORMADO **1.64** · NÃO TEM ENTRETENIMENTO **1.33** · GRAVADO **1.10** |
| `VOO_TIPO` | DIRETO (231.254) | ESCALA **1.14** · CONEXÃO **0.85** |
| `CANCELAMENTO_VOO` | False (307.567) | True **0.96** |

**Fidelidade cresce de forma monótona com o tier.** Contra `AZUL FIDELIDADE`, que é o nível mais
comum: `SEM CADASTRO` 0,85, `TOPAZIO` 1,37, `SAFIRA` 1,46 e `DIAMANTE` 1,86. Quanto mais alto o
tier, maior a chance de o Cliente aparecer como detrator na ordenação do modelo. A Seção 4.2.1 do
documento já registrava esse padrão na exploração, com a leitura de que a expectativa de serviço
acompanha o nível de relacionamento.

**`CANAL_COMPRA` = OTHER tem o maior odds ratio entre os canais e o menor suporte.** São **2.118
linhas**, 0,6% do treino, contra 142.924 do canal de referência. É uma categoria residual medida
sobre pouca gente, e o intervalo em torno dela seria largo. Não sustenta recomendação operacional.

**`CANCELAMENTO_VOO` = True aparece em 0,96 contra False, e esse número não é o efeito do
cancelamento.** Ele é o resíduo de uma coluna cuja informação está repartida em outras seis, como a
Seção 6.4 mostra. Lido sozinho, diria que cancelar não separa quase nada, e o oposto é verdade.

### 6.3. O que o modelo usa, e o que a EDA dizia

**O histórico do Cliente lidera a tabela, e repousa sobre 10% dela.**
`HIST_TAXA_DETRACAO_ANTERIOR` tem odds ratio comparável **4,245**, o maior de todos. Mas o
`n_observado` dela é **34.452 de 341.962 linhas**: a variável só existe para quem já respondeu
antes. O efeito é forte e o suporte é estreito, e as duas coisas precisam ser lidas juntas.

**O atraso na saída aparece acima do atraso na chegada, e isso confirma a EDA.** Na escala
comparável, `ESTATISTICA_ATRASOSAIDA` dá **1,986** ao percorrer os 35 minutos entre p10 e p90, e
`ATRASO_CHEGADA` dá **1,085** nos 29 minutos dela. A Seção 4.2.1, item (c), do documento registra que
`FAIXA_ATRASO` é a discretização de `ESTATISTICA_ATRASOSAIDA` e tem o maior V de Cramér da base,
0,293. A forma contínua é que entrou na allowlist, para não perder a granularidade que a
discretização descarta, mas é a mesma variável. **O modelo e a exploração apontam para o mesmo
lugar.**

Na escala em que a operação decide, o atraso na saída fica assim:

| variável | OR por minuto | 30 min | 1 hora |
|---|---|---|---|
| `ESTATISTICA_ATRASOSAIDA` | 1,0198 | **1,80** | 3,24 |
| `ATRASO_CHEGADA` | 1,0028 | 1,09 | 1,18 |

**`VOO_TIPO` = CONEXÃO parece protetor em 0,844, e não é.** A Seção 6.4 mostra que
`TIPO_ENTRETENIMENTO` = NAO_INFORMADO é a mesma informação, porque a coluna é nula exatamente em
voo com conexão. Os dois se leem juntos: 0,844 x 1,296 = **1,094**. Contra voo direto com
entretenimento ao vivo, que é o par mais comum, a razão é **1,40**. Voo com conexão aparece
**acima** na ordenação do modelo, não abaixo, e isso concorda com a EDA em vez de contradizê-la.

**`N_TRECHOS` fica em 0,950**, e essa continua do lado protetor depois de o atraso entrar na conta.

### 6.4. Três ausências que não são ausência

Três indicadores de ausência desta matriz não medem falta de informação: eles são consequência
determinística de outra coluna. Lidos como se medissem, invertem a leitura do maior efeito do
modelo, e por isso esta seção vem antes de qualquer conclusão operacional.

**As quatro ausências de dados de voo significam "voo cancelado".**
`preprocessamento_nps.aplicar_contrato_temporal_score_pos_viagem` apaga `TEMPO_VOO`,
`ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA` e `N_TRECHOS` em toda linha com `CANCELAMENTO_VOO`
verdadeiro, porque no instante do score de um cancelamento essa informação não existe. No treino o
padrão é exato: **100% de nulos nos cancelados e 0,02% nos demais**. O indicador de ausência de
`TEMPO_VOO`, com odds ratio 3,115, não mede tempo de voo faltando: ele é a coluna de cancelamento
outra vez.

**A ausência de `ANTECEDENCIA_CANCELAMENTO` é a mesma coluna invertida**, porque essa variável só
é preenchida quando houve cancelamento: 0% de nulos nos cancelados e 100% nos demais.

**O cancelamento está repartido em sete colunas colineares**: os quatro indicadores acima, a
ausência de `ANTECEDENCIA_CANCELAMENTO`, `CANCELAMENTO_VOO` = True e `CANCELAMENTO_VOO` = False.
Com penalidade L1, o modo como o solver reparte o peso entre colunas que carregam a mesma
informação é arbitrário.
**Ler 3,115 e 0,384 como efeitos separados não significa nada.** O que tem sentido é somar o
bloco:

| Situação | Odds ratio contra voo não cancelado |
|---|---|
| Cancelado, aviso no mesmo dia | **7,73** |
| Cancelado, aviso na mediana de 10 dias | **6,35** |
| Cancelado, aviso com 48 dias | 2,99 |

Esses números saem do `decision_function` sobre linhas montadas, e **não** de somar coeficientes à
mão. A diferença não é detalhe, porque o `RobustScaler` **centra** cada numérica na mediana do
treino. A coluna de
`ANTECEDENCIA_CANCELAMENTO` vale zero quando a variável está na mediana, não quando vale zero, e o
imputador preenche exatamente essa mediana em todo voo não cancelado. Somar `coeficiente x mediana`
conta duas vezes um deslocamento que o centro já absorveu.

**Esse é o maior efeito do modelo**, acima do histórico do Cliente (4,245). Ler o histórico como o
fator dominante é o erro que a leitura coluna a coluna produz.

**`ANTECEDENCIA_CANCELAMENTO` é medida em dias, e a Seção 6.1 explica por que 0,980 engana.** O
documento registra a unidade na Seção 4.2.1, e a variável vai de 0 a 400 dias, com mediana 10.
Percorrendo 48 dias, `0,9804⁴⁸ = 0,387`. Avisar com antecedência aparece associado a menos
detração, e de forma forte. Isso **concorda com a Hipótese 3** da exploração: 69,2% de detratores
no aviso do mesmo dia contra 24,6% acima de 48 dias.

**O nível `NAO_INFORMADO` de `TIPO_ENTRETENIMENTO` significa "voo com conexão".** A coluna é nula
em 109.698 linhas do treino, e a correspondência com `VOO_TIPO` = CONEXÃO é de **100%**, nos dois
sentidos. Então `NAO_INFORMADO` com 1,296
e `VOO_TIPO` = CONEXÃO com 0,844 também são a mesma informação repartida, e nenhum dos dois se lê
sozinho.

**O que isso deixa para o card 13A.2 (#213) e para a Seção 4.4:** de sete colunas relacionadas a
cancelamento, nenhuma se interpreta isoladamente, e o efeito do bloco é o maior do modelo. A
Regressão Logística com L1 não é a ferramenta certa para atribuir peso entre colunas colineares, e
essa é uma limitação do modelo, não dos dados.

In [ ]:
from odds_ratio import odds_ratio_de_cenario
from preprocessamento_nps import FEATURES_POS_ENCERRAMENTO_JORNADA

preparo_ajustado = pipeline_final.named_steps[PASSO_PREPARO]
imputador = preparo_ajustado.named_transformers_["numericas"].named_steps["imputar"]
medianas = dict(zip(preparo_ajustado.transformers_[0][2], imputador.statistics_))

# A referência tem as colunas de voo já na mediana imputada, para que marcar o
# cancelamento mude só o bloco e nada mais. Sem isso a razão carregaria também a
# diferença entre os valores reais daquela linha e as medianas.
referencia = preparo["x"]["treino"].query("not CANCELAMENTO_VOO").iloc[[0]].copy()
for coluna in FEATURES_POS_ENCERRAMENTO_JORNADA:
    referencia[coluna] = medianas[coluna]

cenarios = {"não cancelado": referencia}
for dias in (0, int(medianas["ANTECEDENCIA_CANCELAMENTO"]), 48):
    cancelado = referencia.copy()
    cancelado["CANCELAMENTO_VOO"] = True
    for coluna in FEATURES_POS_ENCERRAMENTO_JORNADA:
        cancelado[coluna] = np.nan
    cancelado["ANTECEDENCIA_CANCELAMENTO"] = float(dias)
    cenarios[f"cancelado, aviso em {dias} dia(s)"] = cancelado

efeito_cancelamento = pd.DataFrame(
    {"cenário": list(cenarios),
     "odds ratio": odds_ratio_de_cenario(pipeline_final, pd.concat(cenarios.values()))}
)
efeito_cancelamento

## 7. Leitura operacional dos odds ratio (card #213)

A Seção 6 entregou 38 odds ratio com a feature original ao lado. Esta seção traduz os que a área de
Experiência do Cliente consegue usar, em afirmações sobre a operação.

**Antes de qualquer número, o que este texto não afirma.** Odds ratio mede **associação que a
Regressão Logística usa para ordenar**, não causa da detração. Quando a leitura abaixo diz que
trinta minutos de atraso na saída aparecem com chance 1,80 vez maior, ela descreve o que separa
duas respostas na base histórica, e não o que aconteceria se a Azul reduzisse o atraso. Nada aqui
sustenta previsão de efeito de intervenção: para isso seria preciso um desenho que este projeto
não fez.

**Contra o que cada número compara.** O `OneHotEncoder` do contrato é `drop=None`, então a
regressão não tem nível-base: nenhum nível foi omitido da matriz. A referência é declarada na
leitura, e não herdada do modelo. A Seção 6.2 adota o **nível mais frequente de cada variável**, e
é contra ele que as comparações de categoria abaixo estão escritas. Nas variáveis contínuas a
referência é o percurso entre os valores típicos, o p10 e o p90 do treino.

**E odds ratio não é razão de probabilidades.** A prevalência de Detrator no treino é 20,43%, o que
dá uma chance de 0,257. Um odds ratio de 1,80 leva essa chance a 0,462, ou seja, a probabilidade
vai de cerca de **20% para cerca de 32%**, e não de 20% para 37%. Ler o odds ratio como se multiplicasse a
probabilidade exagera o efeito nos dois sentidos: um odds ratio de 0,38 não leva a probabilidade a
7,8%, e sim a cerca de 9%.

### 7.1. O que separa uma resposta detratora das demais

A ordem abaixo é a do **efeito comparável** da Seção 6, e não a do odds ratio cru. A distinção
importa: `ANTECEDENCIA_CANCELAMENTO` aparece em 20º lugar quando se ordena pelo valor bruto e em
2º quando se ordena pelo efeito no intervalo em que ela varia.

**Cancelamento é o fator mais forte, e ele não se lê numa coluna só.** A Seção 6.4 mostra que sete
colunas carregam a mesma informação, porque o contrato apaga os dados de voo quando há
cancelamento. Medindo o bloco pelo `decision_function`, uma resposta de voo cancelado aparece com odds ratio de
**7,73** contra uma de voo não cancelado quando o aviso foi no mesmo dia, **6,35** no aviso da
mediana de 10 dias e **2,99** com 48 dias. É o maior efeito do modelo.

**E avisar antes separa muito.** `ANTECEDENCIA_CANCELAMENTO` é medida em dias, e o odds ratio de
0,980 por dia é o número que engana: percorrendo os 63 dias entre o p10 e o p90, o efeito
comparável é **0,288**. Em 48 dias de antecedência contra aviso no mesmo dia, `0,9804⁴⁸ = 0,387`.

Isso **concorda com a Hipótese 3** da exploração, que mediu 69,2% de detratores no aviso do mesmo
dia contra 24,6% acima de 48 dias.

**O que isso permite e o que não permite.** Dentro do cancelamento, a antecedência do aviso é a
variável que mais separa uma resposta detratora das demais, então ela serve para **priorizar quem
contatar**: entre dois Clientes com voo cancelado, o de aviso mais curto entra antes na fila. O que
o modelo **não** mostra é que antecipar o aviso reduziria a detração. Essa seria uma afirmação
sobre intervenção, e nada aqui a sustenta.

**O histórico do Cliente é o segundo mais forte, e repousa sobre 10% da base.**
`HIST_TAXA_DETRACAO_ANTERIOR` tem efeito comparável **4,245**, mas `n_observado` de **34.452 de
341.962 linhas**: a variável só existe para quem já respondeu antes. Usar a fila de contato
apoiada nela é usar um sinal forte sobre uma minoria.

**O atraso na saída separa mais que o atraso na chegada.** Percorrendo o intervalo típico de cada
uma, 1,986 contra 1,085. Na escala em que a operação decide, meia hora de atraso na saída
corresponde a 1,80.

**Fidelidade cresce de forma monótona com o tier.** Contra `AZUL FIDELIDADE`, o nível mais comum:
`SEM CADASTRO` 0,85, `TOPAZIO` 1,37, `SAFIRA` 1,46 e `DIAMANTE` 1,86. O Cliente de tier mais alto
aparece mais alto na fila com a mesma experiência de voo.

**Voo com conexão aparece acima, não abaixo.** Lido sozinho, `VOO_TIPO` = CONEXÃO dá 0,844 e
pareceria protetor. Mas `TIPO_ENTRETENIMENTO` = NAO_INFORMADO é a mesma informação, porque a coluna
é nula exatamente em voo com conexão, com 100% de correspondência. Contra a referência do par, que
é voo direto com entretenimento ao vivo, o efeito conjunto é de **1,40**.

**Uma leitura que não se sustenta.** `CANAL_COMPRA` = OTHER é o canal de maior odds ratio,
**2,48** contra `AGENCY`, que é o canal mais frequente. Mas ele tem **2.118 linhas**, 0,6% do
treino, contra 142.924 da referência. É categoria residual com pouca massa, e não sustenta
recomendação operacional.

A célula abaixo compõe o efeito por minuto do atraso nas janelas em que a operação raciocina, porque um odds ratio por minuto não é legível sozinho.


In [ ]:
from odds_ratio import efeito_acumulado

JANELAS = {"30 min": 30, "1 hora": 60, "3 horas": 180}


def por_unidade(feature):
    linha = tabela_odds[(tabela_odds["feature_original"] == feature)
                        & (tabela_odds["tipo"] == "numerica")]
    return float(linha.iloc[0]["odds_ratio_por_unidade"])


leitura_atraso = pd.DataFrame([
    {"variável": nome, "OR por minuto": round(por_unidade(nome), 4),
     **{rotulo: round(efeito_acumulado(por_unidade(nome), minutos), 2)
        for rotulo, minutos in JANELAS.items()}}
    for nome in ("ESTATISTICA_ATRASOSAIDA", "ATRASO_CHEGADA")
])
leitura_atraso

| variável | OR por minuto | 30 min | 1 hora | 3 horas |
|---|---|---|---|---|
| `ESTATISTICA_ATRASOSAIDA` | 1,0198 | **1,80** | 3,24 | 34,10 |
| `ATRASO_CHEGADA` | 1,0028 | 1,09 | **1,18** | 1,66 |

Meia hora de atraso na saída aparece com chance 1,80 vez maior de detração; a mesma meia hora na
chegada, 1,09. A explicação plausível é que as duas variáveis andam juntas e o modelo, tendo as
duas disponíveis, concentrou o peso em uma só. Para ordenar a fila, é a saída que carrega o sinal.

A coluna de 3 horas está na tabela de propósito, e a Seção 7.2 explica por quê.

### 7.2. O que este modelo não consegue dizer

As limitações vêm no mesmo texto que os ganhos porque cada uma muda a leitura de um número acima.

**Colinearidade, e não só no cancelamento.** O cancelamento entra no modelo repartido em sete
colunas perfeitamente redundantes, e a penalidade L1 distribui o peso entre elas de forma
arbitrária. O mesmo vale para outros dois pares: `VOO_TIPO` = CONEXÃO com `TIPO_ENTRETENIMENTO` =
NAO_INFORMADO, e os indicadores de ausência de `HIST_DETRATOU_ANTES` e de
`HIST_TAXA_DETRACAO_ANTERIOR`, que valem 1 nas mesmas 307.510 linhas, porque as duas variáveis
faltam juntas. Nenhuma delas se interpreta isoladamente, e só a soma do bloco tem sentido. A
Regressão Logística não é a ferramenta certa para atribuir peso entre colunas que carregam a mesma
informação, e isso é limitação do modelo, não dos dados.

**O efeito é linear no logito, e quebra nos extremos.** A tabela da Seção 7.1 mostra o atraso na
saída em 30 minutos e em 1 hora. Estendendo para 3 horas, o modelo produziria **34,10**, que não
descreve operação real: o odds ratio por minuto é constante e o 180º minuto pesa como o 1º. A
leitura confiável está na faixa com massa, e o p90 de `ESTATISTICA_ATRASOSAIDA` no treino é 35
minutos.

**O modelo não representa interação, e esta base tem uma confirmada.** O efeito do tier e o do
atraso entram somados no logito, nunca multiplicados, então a Regressão Logística não consegue
expressar que o atraso separa mais para o Cliente DIAMANTE do que para o SEM CADASTRO.

Isso não é hipótese em aberto. A **Hipótese 5** do documento confirmou a interação por razão de
verossimilhança entre modelos logísticos, com atraso contínuo e todos os tiers (estatística =
19,06; gl = 6; p = 0,0041) e na comparação direta entre `SEM CADASTRO` e `DIAMANTE` (estatística =
15,15; gl = 1; p < 0,0001).

A ilustração da Seção 4.2.1, que mede 21,3% contra 12,7% em voo pontual e convergência acima de 120
minutos, **não serve como evidência de interação**, e vale registrar por quê: um odds ratio
constante de 1,86, calculado a partir dos próprios 12,7% e 21,3% daquela seção e **não** do modelo,
reproduz o padrão inteiro. Ele leva 12,7% a 21,3% e 71% a 82%, com diferença máxima de 15,4 pontos
no meio da escala. O valor coincide em três casas com o odds ratio de `DIAMANTE` da Seção 7.1, que
é 1,86 contra `AZUL FIDELIDADE`, mas os dois medem coisas diferentes: aquele compara `DIAMANTE`
com `SEM CADASTRO` na exploração, e este compara `DIAMANTE` com `AZUL FIDELIDADE` no modelo. Diferença variável em pontos percentuais é o que um modelo
aditivo no logito já produz sozinho, porque a curva logística não é linear. Só o teste da H5
separa interação de não linearidade.

Ou seja: existe na base um efeito que este modelo, por construção, não consegue representar sem um
termo de interação explícito. É exatamente o terreno em que os ensembles da outra dupla podem
apresentar ganho, e quantificar esse ganho é o que a comparação da Seção 4.4 entrega.

**Dois níveis de `TIER_VIAGEM` nunca entraram no treino.** `AZUL ONE` e `DIAMANTE UNIQUE` só
aparecem a partir de 2025-10-24, depois do corte de validação, então o pré-processador não os
conhece e as 1.353 linhas deles atravessam a pontuação com o bloco de tier zerado. A leitura
monótona de fidelidade vale para os cinco níveis que o treino viu, e **não** se estende aos dois
tiers de topo.

**Nenhum odds ratio aqui tem intervalo de confiança.** Os números saem de um único ajuste sobre a
partição de treino, sem reamostragem. A coluna `n_observado` é o substituto pobre disso: ela diz
quantas linhas sustentam cada valor, e é o que permite separar `CANAL_COMPRA` = OTHER, com 2.118
linhas, de `CANCELAMENTO_VOO` = False, com 307.567.

## 8. Permutation importance sobre blocos colineares (card #215)

A Seção 6.4 mostrou que o cancelamento ocupa sete colunas da matriz. Esta seção mede a
consequência disso para a **permutation importance**, que é a via de explicabilidade post-hoc que
o grupo adotou para os ensembles, no card #191.

A medição está aqui porque o registro de decisão em
[`documents/extras/decisao-explicabilidade-shap.md`](../documents/extras/decisao-explicabilidade-shap.md)
cita estes números, e número citado precisa sair de uma célula que qualquer pessoa reexecute.

O `permutation_importance` embaralha **uma coluna por vez** e mede quanto a métrica piora. Quando
duas colunas carregam a mesma informação, embaralhar uma delas não apaga o sinal, porque a outra
continua ali. O efeito esperado é que cada coluna do bloco pareça pouco importante, mesmo que o
bloco inteiro seja decisivo.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

from preprocessamento_nps import FEATURES_POS_ENCERRAMENTO_JORNADA

REPETICOES = 5
# As seis colunas cruas que carregam o cancelamento: as quatro que o contrato
# apaga quando o voo é cancelado, mais as duas que o nomeiam diretamente.
#
# A lista é explícita e não derivada direto de FEATURES_POS_ENCERRAMENTO_JORNADA
# porque aquela constante é um `frozenset`: a ordem mudaria a cada execução e as
# linhas da tabela abaixo sairiam embaralhadas. A asserção mantém as duas presas
# uma à outra, então mexer no contrato sem mexer aqui levanta erro.
APAGADAS_NO_CANCELAMENTO = ["TEMPO_VOO", "ESTATISTICA_ATRASOSAIDA",
                            "ATRASO_CHEGADA", "N_TRECHOS"]
assert set(APAGADAS_NO_CANCELAMENTO) == set(FEATURES_POS_ENCERRAMENTO_JORNADA)
BLOCO_CANCELAMENTO = [*APAGADAS_NO_CANCELAMENTO,
                      "CANCELAMENTO_VOO", "ANTECEDENCIA_CANCELAMENTO"]

x_val, y_val = preparo["x"]["validacao"], preparo["y"]["validacao"]
base_auc = roc_auc_score(y_val, pipeline_final.predict_proba(x_val)[:, 1])


def queda_ao_permutar(colunas, semente=42):
    """Quanto o ROC-AUC cai ao embaralhar `colunas` com a MESMA permutação de linhas.

    Passar uma coluna reproduz o que `permutation_importance` faz. Passar o bloco
    inteiro é a correção: as colunas perdem a informação juntas, em vez de uma
    sustentar o que a outra perdeu.
    """
    rng = np.random.default_rng(semente)
    quedas = []
    for _ in range(REPETICOES):
        embaralhado = x_val.copy()
        ordem = rng.permutation(len(embaralhado))
        for coluna in colunas:
            embaralhado[coluna] = embaralhado[coluna].to_numpy()[ordem]
        quedas.append(base_auc - roc_auc_score(
            y_val, pipeline_final.predict_proba(embaralhado)[:, 1]))
    return float(np.mean(quedas))


importancia_bloco = pd.DataFrame(
    [{"permutando": coluna, "queda de ROC-AUC": queda_ao_permutar([coluna])}
     for coluna in BLOCO_CANCELAMENTO]
    + [{"permutando": "soma das individuais",
        "queda de ROC-AUC": sum(queda_ao_permutar([c]) for c in BLOCO_CANCELAMENTO)},
       {"permutando": "as seis juntas, mesma permutação",
        "queda de ROC-AUC": queda_ao_permutar(BLOCO_CANCELAMENTO)}]
)
print(f"ROC-AUC na validação, sem permutar: {base_auc:.4f}")
importancia_bloco

| Permutando | Queda de ROC-AUC |
|---|---|
| `TEMPO_VOO` | +0,0215 |
| `ESTATISTICA_ATRASOSAIDA` | +0,0660 |
| `ATRASO_CHEGADA` | +0,0079 |
| `N_TRECHOS` | +0,0002 |
| `CANCELAMENTO_VOO` | **-0,0001** |
| `ANTECEDENCIA_CANCELAMENTO` | +0,0055 |
| soma das individuais | +0,1010 |
| **as seis juntas, mesma permutação** | **+0,1353** |

<div align="center"><sup>Fonte: Autoria própria.</sup></div>

**A coluna que nomeia o cancelamento aparece com importância zero.** Embaralhar `CANCELAMENTO_VOO`
sozinha não muda nada, porque os quatro indicadores de ausência continuam dizendo que o voo foi
cancelado. Pela Seção 6.4, esse mesmo bloco é o **maior efeito do modelo**, com odds ratio de 7,73
no aviso do mesmo dia.

**E somar as importâncias individuais depois não recupera o bloco.** A soma dá 0,1010 contra 0,1353
da permutação conjunta, ou seja, a permutation importance **não é aditiva**. O `scikit-learn`
documenta o efeito em *Misleading values on strongly correlated features*.

Duas consequências, e as duas estão registradas no documento de decisão:

- a leitura de qualquer ranking de permutation importance sobre este conjunto de features precisa
  ser feita **por bloco**, embaralhando as colunas colineares juntas;
- a comparação entre o ranking dos ensembles e os odds ratio da Regressão Logística não é direta
  nesse ponto, e tratá-la como direta levaria à conclusão falsa de que os ensembles não usam o
  cancelamento.

O número acima sai da Regressão Logística, com ROC-AUC. O card #191 mede o ensemble com F2, então
o mecanismo é o mesmo, porque o pré-processador e os indicadores de ausência são os mesmos, mas o
valor que os cards #193, #194 e #251 citarem precisa sair do modelo deles.

## 9. Métricas do modelo final na validação (card #214)

A Seção 4 escolheu os hiperparâmetros por F2 na validação cruzada, e as Seções 6 a 8 leram o
modelo. Falta o que a Seção 4.4 do documento precisa citar: **as três métricas de negócio sobre a
partição de validação**, medidas com a `avaliar()` do card #241, e a métrica de partida contra a
qual o ganho da busca é lido.

Esta seção existe porque o card #214 cita esses números no documento, e número citado precisa sair
de uma célula que qualquer pessoa reexecute.

In [ ]:
from avaliacao import avaliar
from sklearn.metrics import fbeta_score, make_scorer

LIMIAR_PADRAO = 0.5
# Limiar operacional que a Seção 4.3.2.7 do documento declara, derivado da
# capacidade de contato da equipe. Entra aqui só para mostrar por que ele não
# se transporta para este modelo nesta partição.
LIMIAR_SECAO_432 = 0.2934

x_val, y_val = preparo["x"]["validacao"], preparo["y"]["validacao"]

# Partida: o pipeline no padrão da biblioteca, antes de qualquer busca.
partida = criar_pipeline(preparo["preprocessador"])
partida.fit(preparo["x"]["treino"], preparo["y"]["treino"])

def metricas(modelo, limiar=LIMIAR_PADRAO):
    proba = modelo.predict_proba(x_val)[:, 1]
    return avaliar(y_val, (proba >= limiar).astype(int), proba)

comparacao = pd.DataFrame({
    "partida (padrão da biblioteca)": metricas(partida),
    "otimizado (GridSearchCV)": metricas(pipeline_final),
}).loc[["F2", "Sensibilidade", "Precisão Média", "ROC-AUC"]]

# O scoring da busca e o avaliar() do #241 são o mesmo número?
proxy = make_scorer(fbeta_score, beta=2)(pipeline_final, x_val, y_val)
oficial = metricas(pipeline_final)["F2"]
print(f"scoring da busca : {proxy:.10f}")
print(f"avaliar()['F2']  : {oficial:.10f}")
print(f"equivalentes     : {abs(proxy - oficial) < 1e-12}")

# O limiar da Seção 4.3.2.7 aplicado aqui produz que tamanho de fila?
proba_final = pipeline_final.predict_proba(x_val)[:, 1]
print(f"\nfila no limiar {LIMIAR_PADRAO}: {(proba_final >= LIMIAR_PADRAO).sum():,} respostas")
print(f"fila no limiar {LIMIAR_SECAO_432}: {(proba_final >= LIMIAR_SECAO_432).sum():,} respostas")
comparacao

| Métrica | Partida | Otimizado | Meta (Seção 4.1.3) |
|---|---|---|---|
| F2 | 0,2315 | **0,4812** | critério de busca, sem meta |
| Sensibilidade | 0,1976 | **0,4861** | ≥ 0,70 |
| Precisão Média | 0,4996 | **0,5011** | ≥ 0,40 |
| ROC-AUC | 0,7257 | **0,7273** | ≥ 0,75 |

<div align="center"><sup>Fonte: Autoria própria.</sup></div>

**O ganho da busca está inteiro nas métricas que dependem do limiar.** F2 e Sensibilidade mais que
dobram, enquanto Precisão Média e ROC-AUC sobem 0,0015 e 0,0016. É o resultado esperado da Seção
5.2: o `class_weight` foi o único eixo que moveu a métrica de busca, e reponderar a classe desloca
o ponto de corte, não a capacidade de ordenar. Precisão Média e ROC-AUC medem ordenação e são
insensíveis a essa reponderação.

**As duas métricas de limiar valem no limiar padrão de 0,5**, e isso precisa acompanhar o número:
Sensibilidade sem limiar declarado não tem definição.

**O limiar de 0,2934 da Seção 4.3.2.7 não se transporta para cá.** Ele foi derivado para o primeiro
candidato, sobre o conjunto de teste, a partir de uma fila de 9.050 respostas. Aplicado a este
modelo na validação, ele produz uma fila de **38.849 respostas**, cerca de quatro vezes a
capacidade de 50 contatos por dia. Redefinir o limiar por capacidade para cada candidato é o card
13B.1 (#247), e é ele que torna as Sensibilidades comparáveis entre modelos.

**A equivalência do F2** está impressa na célula acima: o `make_scorer(fbeta_score, beta=2)` que a
busca usou e o `avaliar()["F2"]` do card #241 devolvem o mesmo valor. É o que fecha a pendência
registrada na Seção 4.3.